# Homework 03: Python Fundamentals

NumPy operations, loop vs vectorized timing, loading and inspecting `starter_data.csv`, summary statistics, groupby aggregation, and a reusable `get_summary_stats()` utility.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # so `from src.utils import ...` works

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.utils import get_summary_stats, get_group_summary

## 1. NumPy Operations
Create an array and perform elementwise operations.

In [2]:
arr = np.array([2, 4, 6, 8, 10, 12])

print("Array:", arr)
print("Elementwise + 5:", arr + 5)
print("Elementwise * 3:", arr * 3)
print("Elementwise ** 2:", arr ** 2)
print("Sum:", arr.sum(), "| Mean:", arr.mean(), "| Std:", arr.std())

Array: [ 2  4  6  8 10 12]
Elementwise + 5: [ 7  9 11 13 15 17]
Elementwise * 3: [ 6 12 18 24 30 36]
Elementwise ** 2: [  4  16  36  64 100 144]
Sum: 42 | Mean: 7.0 | Std: 3.415650255319866


## 2. Loop vs Vectorized Execution
Same operation (square every element of a large array), two ways, timed.

In [3]:
import time

big = np.arange(1_000_000)

start = time.perf_counter()
loop_result = [x ** 2 for x in big]
loop_time = time.perf_counter() - start

start = time.perf_counter()
vec_result = big ** 2
vec_time = time.perf_counter() - start

print(f"Loop:      {loop_time:.4f}s")
print(f"Vectorized: {vec_time:.4f}s")
print(f"Vectorized was {loop_time / vec_time:.1f}x faster")
assert list(vec_result[:5]) == loop_result[:5], "sanity check: both methods agree"


Loop:      0.1037s
Vectorized: 0.0026s
Vectorized was 40.0x faster


## 3. Load and Inspect Dataset

In [4]:
df = pd.read_csv("data/raw/starter_data.csv", parse_dates=["date"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   category  10 non-null     str           
 1   value     10 non-null     int64         
 2   date      10 non-null     datetime64[us]
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 382.0 bytes


In [5]:
df.head()

,category,value,date
0,A,10,2025-08-01
1,B,15,2025-08-02
2,A,12,2025-08-03
3,B,18,2025-08-04
4,C,25,2025-08-05


## 4. Summary Statistics

In [6]:
summary = get_summary_stats(df)
summary

,value,date
count,10.000000,10
mean,17.600000,2025-08-05 12:00:00
min,10.000000,2025-08-01 00:00:00
25%,12.250000,2025-08-03 06:00:00
50%,14.500000,2025-08-05 12:00:00
75%,23.250000,2025-08-07 18:00:00
max,30.000000,2025-08-10 00:00:00
std,7.381659,NaN


## 5. Groupby Aggregation by Category

In [7]:
group_summary = get_group_summary(df, by="category", agg_col="value")
group_summary

,mean,sum,count
category,,,
A,11.500000,46,4
B,15.666667,47,3
C,27.666667,83,3


## 6. Save Outputs
Summary stats to `data/processed/summary.csv`, plus a bonus bar chart of the groupby result.

In [8]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
summary.to_csv("data/processed/summary.csv")
print("Saved data/processed/summary.csv")

fig, ax = plt.subplots(figsize=(5, 3.5))
group_summary["mean"].plot(kind="bar", ax=ax, color="#2c5cc5")
ax.set_ylabel("mean value")
ax.set_title("Mean value by category")
fig.tight_layout()
fig.savefig("data/processed/summary_by_category.png", dpi=120)
print("Saved data/processed/summary_by_category.png")

Saved data/processed/summary.csv


Saved data/processed/summary_by_category.png


## Notes

- `get_summary_stats()` and `get_group_summary()` live in `src/utils.py` (bonus: moved out of the
  notebook, imported at the top).
- The vectorized NumPy operation beats the pure-Python loop because it pushes the elementwise `**2`
  down into compiled C code operating on a contiguous array, instead of dispatching one Python-level
  operation per element.